# 14 Delta Lake: Interview Q&A

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Consolidate the <code>03_delta_lake</code> folder into spoken answers: the architecture to draw, the core explanations (ACID, time travel, <code>MERGE</code>, <code>OPTIMIZE</code>/<code>VACUUM</code>, layout), rapid-fire questions, scenarios, and a quiz.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Delta Lake questions appear in almost every Databricks interview and in a large part of the Data Engineer Associate exam. Being able to explain <b>how</b> Delta works (the log, commits, files) is what separates "I've used Delta" from "I understand Delta".
</div>

## How to use this notebook

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Rule of thumb</b>
<ol><li>Say each answer out loud before reading the model answer</li><li>Aim for 60–90 seconds for the core explanations, and 1–2 sentences for rapid-fire ones</li><li>Run the quiz cell for random practice</li><li>Log weak answers in <code>interview_prep/mock_interview_log.md</code></li></ol>
</div>

## The picture you must be able to draw

```
                     Delta table
 ┌─────────────────────────────────────────────────────────────┐
 │ _delta_log/                                                 │
 │   000…000.json   protocol, metaData, add f1, add f2         │  each commit = 1 version
 │   000…001.json   remove f1, add f3   (UPDATE)               │  atomic: the file exists or not
 │   000…010.checkpoint.parquet  (state every 10 commits)      │
 │ f1.parquet  f2.parquet  f3.parquet   (data, immutable)      │
 └─────────────────────────────────────────────────────────────┘
 Readers: replay log → live files (+ min/max stats) → skip → read
 Writers: write files → commit N+1 (optimistic concurrency: conflict check, retry or fail)
 Time travel: replay up to version N.  VACUUM: delete unreferenced files older than retention.
```

## Core explanations

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is Delta Lake and why use it instead of plain Parquet?</b><br>
Delta Lake is an open table format that stores data as Parquet files plus a transaction log. The log gives ACID transactions, so failed writes never leave partial data and concurrent writers don't corrupt the table. It also gives schema enforcement and evolution, <code>UPDATE</code>, <code>DELETE</code> and <code>MERGE</code>, time travel and rollback, file-level statistics for data skipping, and a reliable streaming source and sink. Plain Parquet folders have none of that.<br><br>

<b>🎤 2. How does Delta provide ACID guarantees?</b><br>
Every change writes new Parquet files and then commits by atomically creating the next numbered JSON file in <code>_delta_log</code>, listing the files added and removed. Atomicity comes from that single commit file. Readers build a snapshot from committed versions only, which gives isolation. Schema checks and constraints run before the commit, for consistency, and storage makes commits durable. Concurrent writers use optimistic concurrency control: if the version is taken, Delta checks for logical conflicts and retries or fails.<br><br>

<b>🎤 3. How do time travel and RESTORE work?</b><br>
Each version is defined by the log, and removed files stay on storage until <code>VACUUM</code>. So <code>VERSION AS OF</code> or <code>TIMESTAMP AS OF</code> replays the log to that version and reads its files. <code>RESTORE</code> makes an old version current by writing a new commit, so the history is preserved. Time travel is limited by log retention (30 days by default) and file retention (7 days, enforced by <code>VACUUM</code>).<br><br>

<b>🎤 4. Explain MERGE and how you use it for SCD Type 2.</b><br>
<code>MERGE</code> matches a source to a target on a key and, in one transaction, updates or deletes matched rows, inserts new ones, and optionally deletes target rows missing from the source. The source must be unique per key. For SCD2, I stage each changed record twice: once with its key, which matches and closes the current row by setting the end date and the current flag to false, and once with a null merge key, which doesn't match and is inserted as the new current row.<br><br>

<b>🎤 5. What do OPTIMIZE and VACUUM do?</b><br>
<code>OPTIMIZE</code> compacts small files into larger ones, optionally clustering them, and commits a new version with the same data, which fixes the small-files problem. <code>VACUUM</code> permanently deletes data files not referenced by the current version and older than the retention period (7 days by default), which frees storage but ends time travel to versions that needed them. On Databricks, predictive optimization runs both for managed tables.<br><br>

<b>🎤 6. Partitioning vs Z-ordering vs liquid clustering?</b><br>
All three improve data skipping by controlling which values share files. Partitioning splits data into folders by a low-cardinality column and prunes whole folders, which is good only for very large tables. Z-ordering rewrites files so nearby values of high-cardinality columns share files, but it's a full rewrite each time. Liquid clustering, with <code>CLUSTER BY</code>, does the same clustering incrementally, lets you change keys without rewriting, and replaces both. It's the recommended default.
</div>

## Rapid-fire questions

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is in the <code>_delta_log</code>?</b><br>
JSON commit files with <code>add</code>, <code>remove</code>, <code>metaData</code>, <code>protocol</code> and <code>commitInfo</code> actions, plus Parquet checkpoints every 10 commits.<br><br>

<b>🎤 2. Managed vs external table?</b><br>
For a managed table, Unity Catalog manages the storage, and <code>DROP</code> deletes the data. An external table's data lives at your path, and <code>DROP</code> removes only the metadata.<br><br>

<b>🎤 3. Schema enforcement vs schema evolution?</b><br>
Enforcement rejects writes that don't match the schema. Evolution allows changes on purpose, with <code>mergeSchema</code>, <code>overwriteSchema</code>, <code>ALTER TABLE</code>, or <code>MERGE WITH SCHEMA EVOLUTION</code>.<br><br>

<b>🎤 4. <code>mergeSchema</code> vs <code>overwriteSchema</code>?</b><br>
<code>mergeSchema</code> adds new columns on append. <code>overwriteSchema</code> replaces the whole schema on overwrite.<br><br>

<b>🎤 5. How do you rename a column without rewriting data?</b><br>
Enable column mapping (<code>delta.columnMapping.mode = 'name'</code>), then <code>ALTER TABLE ... RENAME COLUMN</code>.<br><br>

<b>🎤 6. What are deletion vectors?</b><br>
Bitmaps that mark deleted rows in existing files, so <code>DELETE</code>, <code>UPDATE</code> and <code>MERGE</code> avoid rewriting whole files.<br><br>

<b>🎤 7. What is the change data feed?</b><br>
Row-level changes per commit (insert, update pre- and post-image, delete), read with <code>readChangeFeed</code> or <code>table_changes()</code> for incremental processing.<br><br>

<b>🎤 8. Which constraints does Delta enforce?</b><br>
<code>NOT NULL</code> and <code>CHECK</code>. Unity Catalog primary and foreign keys are informational.<br><br>

<b>🎤 9. What is an identity column?</b><br>
<code>GENERATED ALWAYS AS IDENTITY</code>: unique, increasing surrogate keys assigned by Delta, possibly with gaps.<br><br>

<b>🎤 10. What does <code>DESCRIBE HISTORY</code> show?</b><br>
Every version with its timestamp, user, operation, parameters and metrics.<br><br>

<b>🎤 11. What does <code>DESCRIBE DETAIL</code> show?</b><br>
Format, location, number of files, size, partition and clustering columns, properties, and protocol versions.<br><br>

<b>🎤 12. Default retention periods?</b><br>
Log retention is 30 days (<code>delta.logRetentionDuration</code>). Deleted file retention is 7 days (<code>delta.deletedFileRetentionDuration</code>).<br><br>

<b>🎤 13. What does <code>OPTIMIZE</code> do to streaming readers?</b><br>
Nothing. Its commits are marked <code>dataChange = false</code>, so streams don't reprocess compacted files.<br><br>

<b>🎤 14. Shallow vs deep clone?</b><br>
A shallow clone copies metadata and references the source's files. A deep clone copies data and metadata, so it's independent.
</div>

## Scenario questions

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. A bad deployment overwrote a Gold table an hour ago. What do you do?</b><br>
Check <code>DESCRIBE HISTORY</code> to find the bad write and the last good version, compare the two versions to confirm, then <code>RESTORE TABLE ... TO VERSION AS OF</code> the good version. Then fix the job and add a check that would have caught it.<br><br>

<b>🎤 2. Queries on a Delta table have become slow over a few weeks. How do you investigate?</b><br>
Check <code>DESCRIBE DETAIL</code> for the file count and average size, because many small files call for <code>OPTIMIZE</code> or auto compaction. Check whether the common filters can skip data, and if not, cluster by those columns. Check the query profile for files and bytes read. Also look at whether deletion vectors have accumulated without <code>OPTIMIZE</code>, and at whether storage and history have grown without <code>VACUUM</code>.<br><br>

<b>🎤 3. Two jobs writing to the same table sometimes fail with <code>ConcurrentAppendException</code>. Why, and what do you do?</b><br>
Both modified overlapping files, so optimistic concurrency detected a conflict. I'd make them touch different partitions or clusters with explicit predicates, run them in sequence, or enable row-level concurrency with deletion vectors on Databricks.<br><br>

<b>🎤 4. A source starts sending a new column. What happens and what should you do?</b><br>
Appends fail because of schema enforcement. If the column is wanted, enable <code>mergeSchema</code> (or schema evolution in Auto Loader and <code>MERGE</code>) for Bronze and Silver, and add it explicitly to Gold when the business needs it.<br><br>

<b>🎤 5. How do you make a daily load idempotent?</b><br>
Use <code>MERGE</code> on the business key with a deduplicated source, or overwrite exactly the batch's data with <code>replaceWhere</code> or dynamic partition overwrite. Then re-running produces the same result.<br><br>

<b>🎤 6. The team must delete a customer's data for GDPR. What steps do you take on Delta?</b><br>
<code>DELETE</code> the rows in every table holding them. With deletion vectors, <code>REORG TABLE ... APPLY (PURGE)</code> rewrites the files. Then <code>VACUUM</code> after the retention period, so old files containing the data are physically removed. Also consider change data feed files and clones.
</div>

## Quiz: random questions

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Picks 5 random questions. Answer each out loud before moving on.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
5 numbered questions. Run again for a new set.
</div>

In [0]:
import random

questions = [
    "What is Delta Lake and why use it instead of Parquet?",
    "How does Delta provide ACID guarantees?",
    "What happens if a write fails halfway?",
    "Explain optimistic concurrency control in Delta.",
    "How do time travel and RESTORE work? What limits them?",
    "MERGE: how do you implement SCD Type 2?",
    "What happens if a MERGE source has duplicate keys?",
    "OPTIMIZE vs VACUUM?",
    "What is data skipping and how do Z-ordering and liquid clustering help it?",
    "When would you still partition a Delta table?",
    "mergeSchema vs overwriteSchema?",
    "What are deletion vectors?",
    "What is the change data feed and how would you use it?",
    "Which constraints does Delta enforce?",
    "Managed vs external tables in Unity Catalog?",
    "A Gold table was overwritten by mistake. What do you do?",
    "Queries on a table got slow over time. How do you investigate?",
    "How do you make a daily load idempotent?",
]
for i, q in enumerate(random.sample(questions, 5), 1):
    print(f"{i}. {q}")

## Prove it: one table, every concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a table and walks through a commit, an update, time travel, <code>RESTORE</code>, <code>OPTIMIZE</code> and the history in a few lines.<br><br>
<b>Why it matters</b><br>Being able to demo the core features quickly is great interview preparation, and a good self-check that you remember the syntax.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A history with <code>CREATE</code>, <code>WRITE</code>, <code>UPDATE</code>, <code>RESTORE</code> and <code>OPTIMIZE</code>, and the restored values.
</div>

In [0]:
DB = "workspace.delta_lake"

In [0]:
t = f"{DB}.delta_demo"
spark.sql(f"CREATE OR REPLACE TABLE {t} (id INT, v STRING) CLUSTER BY (id)")
spark.sql(f"INSERT INTO {t} VALUES (1, 'a'), (2, 'b')")
spark.sql(f"UPDATE {t} SET v = 'oops'")
spark.sql(f"RESTORE TABLE {t} TO VERSION AS OF 1")
spark.sql(f"OPTIMIZE {t}")
spark.table(t).orderBy("id").show()
spark.sql(f"DESCRIBE HISTORY {t}").select("version", "operation").orderBy("version").show()
spark.sql(f"DROP TABLE {t}")

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Which statement about Delta Lake is FALSE?</b><br>
A. Data is stored as Parquet files<br>
B. Each successful write creates a new table version<br>
C. <code>VACUUM</code> creates a new version that can be restored later<br>
D. Old versions can be queried until their files are vacuumed<br>
<details><summary><b>Show answer</b></summary><b>C.</b> <code>VACUUM</code> deletes files and doesn't create a restorable data version.</details><br><br>

<b>Q2. A data engineer wants to remove rows from a target table that no longer appear in a full snapshot source, in the same operation as the upsert. Which clause should they add to the <code>MERGE</code>?</b><br>
A. <code>WHEN NOT MATCHED THEN DELETE</code><br>
B. <code>WHEN NOT MATCHED BY SOURCE THEN DELETE</code><br>
C. <code>WHEN MATCHED AND source IS NULL THEN DELETE</code><br>
D. <code>DELETE WHERE NOT EXISTS</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q3. Which feature lets a downstream job read only the rows that changed in a Delta table since a given version?</b><br>
A. Time travel<br>
B. Change data feed<br>
C. Liquid clustering<br>
D. <code>DESCRIBE HISTORY</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q4. A table is queried by <code>customer_id</code> (high cardinality) and <code>event_date</code>. The team wants good data skipping on both and the ability to change keys later. What should they use?</b><br>
A. Partition by both columns<br>
B. <code>ZORDER BY (customer_id, event_date)</code> once<br>
C. <code>CLUSTER BY (customer_id, event_date)</code><br>
D. Bucketing<br>
<details><summary><b>Show answer</b></summary><b>C.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Draw the Delta table picture from memory and compare it with the one above</li><li>Record yourself answering the six core explanations, 60–90 seconds each</li><li>Run the quiz three times and answer every question out loud</li><li>Re-run the "prove it" cell from memory in a new notebook, without looking at the code</li><li>Add your weakest two answers to <code>interview_prep/mock_interview_log.md</code></li></ol>
</div>

In [0]:
# Notes, or your own experiments while practising.

## Summary

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Delta = Parquet + transaction log. Each commit is atomic and is one version</li><li>Readers see consistent snapshots. Writers use optimistic concurrency</li><li>Enforcement protects the schema. Evolution changes it on purpose</li><li><code>UPDATE</code>/<code>DELETE</code>/<code>MERGE</code> rewrite files or use deletion vectors. <code>MERGE</code> powers upserts and SCD2</li><li>Time travel and <code>RESTORE</code> within retention. <code>VACUUM</code> ends it for deleted files</li><li><code>OPTIMIZE</code> fixes small files. Liquid clustering (or Z-order) makes data skipping effective</li><li>CDF enables incremental pipelines. Constraints protect tables from every writer</li></ul>
</div>

| Lesson | Topic |
|---|---|
| 01 | Fundamentals, managed vs external |
| 02 | Creating and querying tables |
| 03 | ACID and the log |
| 04–05 | Schema enforcement and evolution |
| 06–07 | UPDATE, DELETE, MERGE, SCD |
| 08 | Time travel, RESTORE, CLONE |
| 09 | OPTIMIZE and VACUUM |
| 10 | Partitioning, Z-order, liquid clustering |
| 11 | Change data feed |
| 12 | Constraints, generated and identity columns |
| 13 | Medallion with Delta |

## Git commit

```
git add 03_delta_lake/14_delta_interview_qa.ipynb
git commit -m "add 03_14 delta interview qa notebook"
```